In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!ls scripts


/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 180 (delta 51), reused 163 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 137.22 KiB | 17.15 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/RiskAware-Complaints-Engine
error_analysis_category.py  prepare_lstm_data.py    tune_category_optuna.py
evaluate_category.py	    train_category.py	    tune_category.py
prepare_data.py		    train_lstm_category.py


In [19]:
import os, glob, shutil,pathlib,yaml, torch, zipfile
from google.colab import files,drive
from pathlib import Path

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ROOT = "/content/RiskAware-Complaints-Engine"
os.makedirs(f"{ROOT}/data/raw", exist_ok=True)
matches = glob.glob("/content/drive/MyDrive/**/cfpb_complaints.csv", recursive=True)
print("matches:", matches)

if not matches:
    raise FileNotFoundError("cfpb_complaints.csv не найден в MyDrive")

SRC = matches[0]
DST = f"{ROOT}/data/raw/cfpb_complaints.csv"
shutil.copy(SRC, DST)

print("copied:", SRC, "->", DST)
print("exists:", os.path.exists(DST))

matches: ['/content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv']
copied: /content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv -> /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv
exists: True


In [14]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.01 KiB | 345.00 KiB/s, done.
From https://github.com/Nusultan11/RiskAware-Complaints-Engine
   5d6d712..51252d0  master     -> origin/master
Updating 5d6d712..51252d0
error: Your local changes to the following files would be overwritten by merge:
	configs/category.yaml
Please commit your changes or stash them before you merge.
Aborting


In [15]:
cfg_path = pathlib.Path("/content/RiskAware-Complaints-Engine/configs/category.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["stacks"]["bilstm"]["max_length"] = 256
cfg["stacks"]["bilstm"]["embedding_dim"] = 128
cfg["stacks"]["bilstm"]["hidden_dim"] = 128
cfg["stacks"]["bilstm"]["batch_size"] = 256
cfg["stacks"]["bilstm"]["epochs"] = 6

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("updated")

updated


In [12]:
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 1
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

epoch=1 train_loss=3.1807 val_macro_f1=0.0230 val_accuracy=0.2992
LSTM category training completed.
Best epoch: 1
Val Macro-F1: 0.0230
Test Macro-F1: 0.0253
Artifacts saved to: artifacts/category_lstm
Metrics saved to: reports/metrics
{
  "macro_f1": 0.02302730605863733,
  "accuracy": 0.2991565897774616
}{
  "macro_f1": 0.02532048740824743,
  "accuracy": 0.30622204148027654
}

In [16]:
!PYTHONPATH=src python scripts/prepare_lstm_data.py
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 6
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

LSTM preprocessing completed.
Saved tensors to: artifacts/lstm_preprocessing
epoch=1 train_loss=3.1959 val_macro_f1=0.0228 val_accuracy=0.3016
epoch=2 train_loss=2.4470 val_macro_f1=0.0449 val_accuracy=0.3757
epoch=3 train_loss=2.1203 val_macro_f1=0.0709 val_accuracy=0.4188
epoch=4 train_loss=1.8972 val_macro_f1=0.1068 val_accuracy=0.4501
epoch=5 train_loss=1.7358 val_macro_f1=0.1273 val_accuracy=0.4690
epoch=6 train_loss=1.5943 val_macro_f1=0.1427 val_accuracy=0.4754
LSTM category training completed.
Best epoch: 6
Val Macro-F1: 0.1427
Test Macro-F1: 0.1516
Artifacts saved to: artifacts/category_lstm
Metrics saved to: reports/metrics
{
  "macro_f1": 0.14266128291877864,
  "accuracy": 0.47535819530535517
}{
  "macro_f1": 0.15160057211993377,
  "accuracy": 0.47956486376575846
}

In [20]:
bundle = Path("lstm_v1_bundle.zip")
paths = [
    Path("artifacts/category_lstm"),
    Path("artifacts/lstm_preprocessing/metadata.json"),
    Path("artifacts/lstm_preprocessing/vocab.json"),
    Path("reports/metrics/category_lstm_metrics_val.json"),
    Path("reports/metrics/category_lstm_metrics_test.json"),
    Path("configs/category.yaml"),
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.is_file():
            z.write(p, arcname=str(p))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f))

print("created:", bundle.resolve())

created: /content/RiskAware-Complaints-Engine/lstm_v1_bundle.zip


In [21]:
files.download("/content/RiskAware-Complaints-Engine/lstm_v1_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>